# Module 3: High-Performance Data Processing (Generators & Decorators)

**Concepts:** Lazy evaluation · `yield` · generators · decorators

**Who this is for:** You are new to Python. This module uses **stories and real-world pictures** first, then short code you can run and change.

**Why Platform / MLOps care:** Log files and metric streams can be **huge**. You must not load all of them into RAM. And you must not copy-paste **retry** and **logging** into every function — that is what **decorators** fix.

---

### How to use this notebook

1. Read the **markdown** cell (the explanation).
2. Run the **code** cell right below it (`Shift+Enter`).
3. Compare output to the **What you should see** line.
4. Do exercises before opening Solutions.

**Remember for the whole module:** Generators = **one piece at a time**. Decorators = **extra rules wrapped around** a function once.


## 1. Lazy vs eager — do you load everything at once?

**What is this?** Two ways to handle many items (log lines, metrics, file rows).

**Real life (eager):** Photocopy **every page** of a 5,000-page manual before you read page 1. Your backpack explodes. In Python: `readlines()` on a giant file → uses lots of **memory**.

**Real life (lazy):** Read **one page**, decide if you care, turn the page. In Python: a **generator** gives you **one item at a time** when your `for` loop asks for the next.

| Style | Plain English | Infra example |
|-------|---------------|---------------|
| **Eager** | Build the full list now | Load entire log into a list |
| **Lazy** | Produce on demand | Stream log lines one by one |

**When you run the next cell, look for:** Two ways to get `[1, 2, 3]` — one is a list already, one is built step-by-step via `yield`.

**Remember:** Lazy = **conveyor belt**, not **warehouse full of boxes**.


In [ ]:
def first_three_eager():
    # Eager: whole list exists immediately in memory
    return [1, 2, 3]


def first_three_lazy():
    # Lazy: give one number, pause, give next when asked
    yield 1
    yield 2
    yield 3


print("eager:", first_three_eager())
print("lazy as list:", list(first_three_lazy()))


## 2. `yield` — pause the function and hand out one value

**What is `yield`?** Like a chef who plates **one dish** and waits until the waiter asks for the next — the recipe is **not** finished in one go.

**Real life:** WhatsApp scroll — messages appear as you go; the app does not need all future messages in memory at once.

**In a log script:** `yield line` sends one log line to the loop; the function **pauses** until the loop asks again.

**When you run the next cell, look for:** Only lines containing `ERROR` print as `ALERT: ...`.

**Remember:** `return` = done forever. `yield` = here is one item; **I can continue later**.


In [ ]:
def read_fake_log():
    # Pretend this is a huge file — we still emit one line at a time
    for line in ["INFO start", "ERROR timeout", "INFO done"]:
        yield line


for line in read_fake_log():
    if "ERROR" in line:
        print("ALERT:", line)


## 3. Generator expression — a short lazy filter

**What is it?** A one-line lazy version of “loop and filter.”

Syntax: `(item for item in stuff if condition)` — parentheses, not square brackets.

**Real life:** Security guard with a checklist: only visitors with a red badge get through — you do not first build a list of all visitors in the building.

**Platform:** From thousands of lines, keep only those with `ERROR` — without storing all lines in a list.

**When you run the next cell, look for:** Two error lines printed.

**Remember:** `[ ... ]` = eager list. `( ... for ... )` = lazy generator.


In [ ]:
lines = ["ok", "ERROR disk", "ok", "ERROR net"]
errors = (ln for ln in lines if "ERROR" in ln)
for e in errors:
    print(e)


## 4. Small pipeline — parse, then filter (like metrics in Prometheus text)

**What is this?** Stage 1: split `cpu=1.2` into name + value. Stage 2: keep high CPU values. Done **lazily**.

**Real life (MLOps / monitoring):** Raw text → extract `cpu=1.7` → keep if CPU > 1.5. Same idea as grep + awk, but in Python you can chain steps.

**Do not panic** if the generator expression looks long — read it as: *for each line, split; if name is cpu and value high, keep value*.

**When you run the next cell, look for:** A list with one float (high cpu values only).

**Remember:** Pipelines = **parse → filter → act**, often one item at a time.


In [ ]:
def split_fields(line: str):
    name, value = line.split("=", maxsplit=1)
    yield name.strip(), value.strip()


raw = ["cpu=1.2", "mem=80", "cpu=2.0"]
high_cpu = (
    val for name, val in (split_fields(l) for l in raw)
    for _ in [0] if name == "cpu" and float(val) > 1.5
)
print("high cpu readings:", list(high_cpu))


## 5. Decorators — one rule for many functions

**What is a function?** A named recipe: inputs → work → result. Example: scale replicas to 3.

**What is a decorator?** A **frame** around that recipe. The recipe stays the same; the frame adds steps **before and after** (log, retry, check token).

**Real life (school):** Every essay must have a cover sheet with name and date. You do not rewrite each essay — **one rule for all**. In Python, `@cover_sheet` on a function = that rule.

**Real life (DevOps):** nginx in front of an app adds logging and TLS without changing app code. A decorator does that **inside your Python file** for functions.

**What does `@log_calls` do?**

1. Python defines your function `scale`.
2. `@log_calls` passes `scale` into `log_calls`, which builds `wrapper`.
3. The name `scale` now points to `wrapper`. When you call `scale(3)`, the wrapper logs, runs real `scale`, logs again, returns the result.

**When you run the next cell, look for:** A line starting with `CALL scale` **before** the result, and `DONE scale` **after**.

**Remember:** Decorator = **write once**, attach with `@...` on many functions — no copy-paste logging.


In [ ]:
def log_calls(fn):
    def wrapper(*args, **kwargs):
        print(f"CALL {fn.__name__}{args}")
        result = fn(*args, **kwargs)
        print(f"DONE {fn.__name__} -> {result!r}")
        return result
    return wrapper


@log_calls
def scale(replicas: int) -> str:
    return f"scaled to {replicas}"


scale(3)


## 6. Decorator with settings — automatic retry (like CI or kubectl)

**What is this?** Same frame idea, but the frame says: **if it fails, try again** (API timeout, pod not ready).

**Real life:** You knock on a door; if nobody answers, wait 5 seconds and knock again — up to 3 times. You do not rewrite the “knock” action each time; **retry logic wraps** it.

**Platform:** GitLab API or health check flaps — retry with sleep instead of failing the whole deploy on first glitch.

**When you run the next cell, look for:** `attempt 1 failed` then success on attempt 2.

**Remember:** `@retry(...)` is a decorator factory — it **returns** the decorator that wraps your function.


In [ ]:
import time


def retry(times: int = 3, delay: float = 0.1):
    def decorator(fn):
        def wrapper(*args, **kwargs):
            last_err = None
            for attempt in range(1, times + 1):
                try:
                    return fn(*args, **kwargs)
                except Exception as exc:
                    last_err = exc
                    print(f"attempt {attempt} failed: {exc}")
                    time.sleep(delay)
            raise last_err
        return wrapper
    return decorator


attempt = 0


@retry(times=3, delay=0.05)
def flaky_check() -> str:
    global attempt
    attempt += 1
    if attempt < 2:
        raise TimeoutError("not ready")
    return "up"


print(flaky_check())


## 7. `functools.wraps` — keep the function's real name

**What is the problem?** After decorating, Python might think your function is called `wrapper` — bad for logs and debugging.

**Fix:** `@wraps(fn)` inside the decorator copies the original name/docstring to the wrapper.

**When you run the next cell, look for:** `[deploy] rolling update` and `run.__name__` should print `run`, not `wrapper`.

**Remember:** Real projects almost always use `wraps` inside decorators.


In [ ]:
from functools import wraps


def tag(name: str):
    def deco(fn):
        @wraps(fn)
        def wrapper(*a, **k):
            print(f"[{name}]", end=" ")
            return fn(*a, **k)
        return wrapper
    return deco


@tag("deploy")
def run() -> None:
    print("rolling update")


run()
print("function name is still:", run.__name__)


## 8. Capstone — lazy metrics + logging decorator together

**Story:** You have a list of fake metric lines (`cpu=0.9`, `cpu=1.7`, ...). You want:

1. **Lazy parse** — walk lines one by one, `yield` each `(name, value)` pair (no giant dict of everything).
2. **Find max CPU** — normal function logic.
3. **Logging decorator** — print CALL/DONE around that function so your deploy log shows who ran what.

This cell includes `log_calls` again so it runs even if you jump here (still better to run the notebook top-to-bottom once).

**When you run the next cell, look for:** CALL line, then max CPU `1.7`, then DONE line.

**Remember one sentence:** Stream with **yield**, wrap behaviour with **@decorator** — that is how you stay fast and tidy on real logs and APIs.


In [ ]:
def log_calls(fn):
    def wrapper(*args, **kwargs):
        print(f"CALL {fn.__name__}{args}")
        result = fn(*args, **kwargs)
        print(f"DONE {fn.__name__} -> {result!r}")
        return result
    return wrapper


def metric_lines(events: list[str]):
    for e in events:
        if "=" in e:
            k, v = e.split("=", 1)
            yield k.strip(), float(v.strip())


@log_calls
def max_metric(events: list[str], key: str) -> float:
    vals = [v for k, v in metric_lines(events) if k == key]
    return max(vals) if vals else 0.0


events = ["cpu=0.9", "cpu=1.7", "mem=70"]
print("max cpu:", max_metric(events, "cpu"))


---

# Exercises

Try yourself in the `# TODO` cells. Hints are in the markdown above each cell.


**Ex 1 — Countdown generator**

Write `countdown(n)` that **yields** `n`, then `n-1`, ... down to `1`.

**Hint:** `while` loop + `yield` + decrease counter.


In [ ]:
# TODO: def countdown(n): ...


**Ex 2 — Squares 1..5 (generator expression)**

Set `g = ( ... )` so `list(g)` is `[1, 4, 9, 16, 25]`. Run `list(g)` only once (generators empty after use).


In [ ]:
# TODO


**Ex 3 — Decorator `upper_result`**

If the function returns a **string**, return `.upper()`; otherwise return unchanged.

Test with a function returning `'prod'`.


In [ ]:
# TODO


**Ex 4 — Last n lines**

Generator `tail(lines, n)` yields the **last n** lines (using a list buffer is OK for this exercise).


In [ ]:
lines = ["l1", "l2", "l3", "l4"]
# TODO


---
# Solutions (after you tried)


In [ ]:
def countdown(n: int):
    while n >= 1:
        yield n
        n -= 1


print(list(countdown(3)))


In [ ]:
g = (x * x for x in range(1, 6))
print(list(g))


In [ ]:
def upper_result(fn):
    def wrapper(*a, **k):
        r = fn(*a, **k)
        return r.upper() if isinstance(r, str) else r
    return wrapper


@upper_result
def env() -> str:
    return "prod"


print(env())


In [ ]:
def tail(lines, n: int):
    buf = list(lines)
    for line in buf[-n:]:
        yield line


lines = ["l1", "l2", "l3", "l4"]
print(list(tail(lines, 2)))


**Next module:** Module 4 — when things go wrong (`try/except`) and guaranteed cleanup (`with`).
